# θ₃ in terms of (a, b, c, d, θ₂), and the foot function C_k(v)

**Goal.** Synthesis Fundamentals §4.2 gives the foot of a candidate leg in the body frame as

$$
\mathbf C(\theta_2;\mathbf v) = \mathbf H + e^{j\theta_1}\Big[a\,e^{j\theta_2} + p\,e^{j(\theta_3+\delta)}\Big],
\qquad \theta_{2,k} = \theta_0 - \Delta\theta_k,
$$

with design vector $\mathbf v = (a, b, c, d, p, \delta, \theta_1, \theta_0, h_X, h_Y)$.

Every symbol on the right is either in $\mathbf v$ or is the crank angle, **except $\theta_3$**, the coupler angle.
This notebook writes $\theta_3$ explicitly as a function of $a, b, c, d, \theta_2$ (plus a branch sign), so that
$\mathbf C_k$ is a closed-form function of $\mathbf v$ alone. It then checks the result against the best leg
published in SF §4.6.

Conventions (same as KF / SF / the toolkit): the four-bar frame has $O_2$ at the origin and $O_4$ at $(d, 0)$.
All angles are counter-clockwise from the ground line $O_2\to O_4$. Lengths are in mm.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

D2R = np.pi / 180

## 1. The task: seven timed stance targets (SF §4.1)

The stance is half a crank turn with a 70 mm stride, checked every 30° of crank:

$$\mathbf T_k = \Big(35 - \tfrac{70}{180^\circ}\,\Delta\theta_k,\; 0\Big),\qquad \Delta\theta_k = 0^\circ, 30^\circ, \dots, 180^\circ .$$

> **Note on `opt.py`.** Line 7 there has `T_k = [(-50/180)*dtheta_k, zeros]`. That has a 50 mm stride and no
> `+35` offset, so the targets run from 0 to −50 instead of being centred on the origin. Also, the comment on
> line 5 says `[0, 1, ... 7]`, but `np.arange(1, 8)` gives 1…7. If you meant a different stride on purpose,
> ignore this. The offset still matters, because SF puts the origin at mid-stance.

In [ ]:
dth = np.arange(0, 181, 30) * D2R               # crank travel since start of stance, 7 values [rad]
T = (35 - 70 / np.pi * dth) + 0j                # targets as complex numbers X + jY  [mm]
print(np.round(T.real, 2))

## 2. Why θ₃ is the hard part

The vector loop (SF §3.2, Step 1) is

$$a\,e^{j\theta_2} + b\,e^{j\theta_3} - c\,e^{j\theta_4} - d = 0 .$$

Given $\theta_2$, this is one complex equation (two real ones) in **two** unknowns, $\theta_3$ and $\theta_4$. We
want only $\theta_3$, so we eliminate $\theta_4$. Freudenstein eliminated $\theta_3$ by squaring away the coupler (SF §3.2).
We do the mirror image and square away the **rocker**, which is the same trick: *write a length squared and the
unknown direction drops out.*

The cleanest way to see it is with one extra arrow, the **diagonal** from the crank tip $A$ to the rocker pivot $O_4$.

### 2a. The diagonal

The crank tip is $\mathbf A = a\,e^{j\theta_2}$. The arrow from $A$ to $O_4$ is

$$\mathbf s = d - a\,e^{j\theta_2}, \qquad f = |\mathbf s| = \sqrt{a^2 + d^2 - 2ad\cos\theta_2}, \qquad
\phi = \arg\mathbf s = \operatorname{atan2}(-a\sin\theta_2,\; d - a\cos\theta_2).$$

Both $f$ and $\phi$ depend only on $a, d, \theta_2$. They are known once the crank angle is known.

### 2b. The triangle A–B–O₄

Points $A$, $B$, $O_4$ form a triangle with sides $b$ (coupler, $A\to B$), $c$ (rocker, $O_4\to B$) and $f$
(diagonal, $A\to O_4$). The law of cosines gives the angle $\psi$ at $A$, between the diagonal and the coupler:

$$c^2 = b^2 + f^2 - 2bf\cos\psi \quad\Longrightarrow\quad
\psi = \arccos\!\left(\frac{b^2 + f^2 - c^2}{2\,b\,f}\right).$$

This is the step that removes $\theta_4$: the rocker enters only through its **length** $c$.

### 2c. θ₃

The coupler points along the diagonal, turned by $\pm\psi$:

$$\boxed{\;\theta_3(a,b,c,d,\theta_2) = \operatorname{atan2}\!\big(-a\sin\theta_2,\; d - a\cos\theta_2\big)
\;+\; \sigma\,\arccos\!\left(\frac{b^2 + a^2 + d^2 - 2ad\cos\theta_2 - c^2}{2\,b\,\sqrt{a^2 + d^2 - 2ad\cos\theta_2}}\right),
\qquad \sigma = \pm 1\;}$$

- **σ is the assembly (branch).** The triangle can be folded to either side of the diagonal: $B$ above or below the
  line $A O_4$. These are the two assemblies of KF Part 4. You choose σ once for a design and keep it for the
  whole turn. For the walking leg in SF §4.6, **σ = +1** is the one that walks (shown in §5 below).
- **When it fails.** If $\left|\frac{b^2+f^2-c^2}{2bf}\right| > 1$, no triangle exists: the links cannot reach at that
  $\theta_2$. That happens only when $f < |b-c|$ or $f > b+c$. The "crank must turn" (Grashof crank-rocker) rule in
  the residuals keeps the optimizer away from such candidates. The code returns `NaN` there so the failure can't go unnoticed.
- **Always use `atan2`, not `arctan`.** The diagonal can point into any quadrant.

Once you have θ₃, θ₄ follows from the same loop if you ever need it (e.g. for the transmission angle):
$c\,e^{j\theta_4} = a e^{j\theta_2} + b e^{j\theta_3} - d$, so $\theta_4 = \arg(a e^{j\theta_2} + b e^{j\theta_3} - d)$
and $\mu = |\theta_4 - \theta_3|$ folded into $[0^\circ, 90^\circ]$.

In [ ]:
def theta3(a, b, c, d, th2, sigma=+1):
    # Coupler angle theta3 of a four-bar (O2 at origin, O4 at (d, 0)), vectorized over th2 [rad].
    # sigma = +1 / -1 picks the assembly. Returns NaN where the linkage cannot assemble.
    s = d - a * np.exp(1j * th2)                   # diagonal arrow A -> O4
    f = np.abs(s)
    cos_psi = (b**2 + f**2 - c**2) / (2 * b * f)   # law of cosines in triangle A-B-O4
    psi = np.arccos(np.where(np.abs(cos_psi) <= 1, cos_psi, np.nan))
    return np.angle(s) + sigma * psi


def theta4(a, b, c, d, th2, th3):
    # Rocker angle from the loop, once theta3 is known.
    return np.angle(a * np.exp(1j * th2) + b * np.exp(1j * th3) - d)

## 3. Cross-check: Norton's half-angle formula (what KF and the toolkit use)

Norton (ch. 4) solves the same loop a different way. Eliminate θ₄, substitute
$\cos\theta_3 = \frac{1-t^2}{1+t^2}$, $\sin\theta_3 = \frac{2t}{1+t^2}$ with $t = \tan(\theta_3/2)$, and solve a quadratic:

$$K_1 = \frac{d}{a},\quad K_4 = \frac{d}{b},\quad K_5 = \frac{c^2 - d^2 - a^2 - b^2}{2ab}$$
$$D = \cos\theta_2 - K_1 + K_4\cos\theta_2 + K_5,\qquad E = -2\sin\theta_2,\qquad F = K_1 + (K_4 - 1)\cos\theta_2 + K_5$$
$$\theta_3 = 2\,\operatorname{atan2}\!\left(-E \pm \sqrt{E^2 - 4DF},\; 2D\right).$$

The formula looks different, but it describes the same two assemblies. The cell below checks that the two methods
agree over a full turn and shows which ± goes with which σ.

In [ ]:
def theta3_norton(a, b, c, d, th2, sign=+1):
    K1, K4, K5 = d / a, d / b, (c**2 - d**2 - a**2 - b**2) / (2 * a * b)
    D = np.cos(th2) - K1 + K4 * np.cos(th2) + K5
    E = -2 * np.sin(th2)
    F = K1 + (K4 - 1) * np.cos(th2) + K5
    return 2 * np.arctan2(-E + sign * np.sqrt(E**2 - 4 * D * F), 2 * D)


def wrap(x):                                       # angle difference folded into (-pi, pi]
    return np.angle(np.exp(1j * x))


a, b, c, d = 21.2, 63.5, 29.0, 66.5                # the SF section 4.6 best leg
th2 = np.linspace(0, 2 * np.pi, 721)
for sigma in (+1, -1):
    for sign in (+1, -1):
        err = np.max(np.abs(wrap(theta3(a, b, c, d, th2, sigma) - theta3_norton(a, b, c, d, th2, sign))))
        print(f"sigma={sigma:+d}  vs  Norton {'+' if sign > 0 else '-'}sqrt : max diff = {err:.2e} rad")

**Result:** σ = +1 is Norton's **−√** root, and σ = −1 is the **+√** root. They agree to machine precision.
If your toolkit's `FourBar` uses branch labels `+1/−1`, check which one matches by comparing a single θ₃ value.
SF §4.6 calls the walking leg's assembly the "open assembly" / branch +1 (`min_transmission_angle(+1)`).

### Loop-closure and rigidity checks

Plug θ₃ and θ₄ back into the loop. It should close to round-off, and $|B - A|$ should equal $b$ at every angle.

In [ ]:
for sigma in (+1, -1):
    t3 = theta3(a, b, c, d, th2, sigma)
    t4 = theta4(a, b, c, d, th2, t3)
    A = a * np.exp(1j * th2)
    B = A + b * np.exp(1j * t3)
    loop = np.abs(A + b * np.exp(1j * t3) - c * np.exp(1j * t4) - d)
    print(f"sigma={sigma:+d}: max loop residual = {loop.max():.1e} mm, "
          f"max ||B-A|-b| = {np.abs(np.abs(B - A) - b).max():.1e}, "
          f"max ||B-O4|-c| = {np.abs(np.abs(B - d) - c).max():.1e}")

## 4. C_k fully defined in terms of v

Substituting θ₃ into the foot equation, each piece is now a function of $\mathbf v$ and the crank travel $\Delta\theta_k$ only:

| Term | Meaning | Depends on |
|---|---|---|
| $\theta_{2,k} = \theta_0 - \Delta\theta_k$ | crank angle at target $k$ (motor turns **clockwise**, so subtract) | $\theta_0$ |
| $a\,e^{j\theta_{2,k}}$ | arrow $O_2\to A$, crank tip, in the four-bar frame | $a, \theta_0$ |
| $\theta_3(a,b,c,d,\theta_{2,k})$ | coupler direction, from §2c | $a,b,c,d,\theta_0$ |
| $p\,e^{j(\theta_3+\delta)}$ | arrow $A\to$ foot; the foot sits at distance $p$ and angle $\delta$ from line $AB$ | $p,\delta$ + above |
| $e^{j\theta_1}[\cdots]$ | rotate the four-bar frame (x along $O_2O_4$) into the body frame | $\theta_1$ |
| $\mathbf H = h_X + j h_Y$ | translate to the hip $O_2$'s place on the body | $h_X, h_Y$ |

$$
\boxed{\;\mathbf C_k(\mathbf v) = (h_X + j h_Y) + e^{j\theta_1}\Big[a\,e^{j(\theta_0-\Delta\theta_k)}
+ p\,e^{j\left(\theta_3(a,b,c,d,\;\theta_0-\Delta\theta_k)\,+\,\delta\right)}\Big]\;}
$$

Note: only the crank tip and the coupler point are needed. $c$ and $d$ never appear as separate arrows. They enter
only through θ₃, which is how the rocker constrains where the coupler points.

In [ ]:
def foot(v, dth, sigma=+1):
    # Foot position C (complex, body frame, mm) of design v at crank travels dth [rad].
    # v = (a, b, c, d, p, delta, theta1, theta0, hX, hY), angles in radians.
    a, b, c, d, p, delta, th1, th0, hX, hY = v
    th2 = th0 - dth                                    # clockwise motor
    th3 = theta3(a, b, c, d, th2, sigma)
    C = (hX + 1j * hY) + np.exp(1j * th1) * (a * np.exp(1j * th2) + p * np.exp(1j * (th3 + delta)))
    return C, th2, th3

## 5. Verify against the best leg of SF §4.6

The published design is
$a,b,c,d = 21.2, 63.5, 29.0, 66.5$ mm, $p = 82.3$ mm, $\delta = 65.2^\circ$, $\theta_1 = 139.7^\circ$,
$\theta_0 = -91.5^\circ$, $\mathbf H = (50.0, 62.7)$ mm. SF reports a stance rms miss of **2.7 mm** and a worst miss of **4.4 mm**.
If our $\mathbf C_k$ is right, at least one branch has to reproduce those. (The published v is rounded to 0.1, so
expect small differences.)

In [ ]:
v_best = np.array([21.2, 63.5, 29.0, 66.5, 82.3, 65.2 * D2R, 139.7 * D2R, -91.5 * D2R, 50.0, 62.7])

for sigma in (+1, -1):
    C, _, _ = foot(v_best, dth, sigma)
    miss = np.abs(C - T)
    print(f"sigma={sigma:+d}: stance rms miss = {np.sqrt(np.mean(miss**2)):6.2f} mm, worst = {miss.max():6.2f} mm")

# Swing clearance, sampled at every degree of the swing half-turn (dth from 180 to 360 deg)
dth_sw = np.arange(180, 361, 1) * D2R
Cw, _, _ = foot(v_best, dth_sw, +1)
mid = np.abs(dth_sw - 270 * D2R) <= 20 * D2R
print(f"sigma=+1: swing lift (highest foot point) = {Cw.imag.max():.1f} mm   (SF reports 15.8 mm)")
print(f"          lowest point within 20 deg of mid-swing = {Cw.imag[mid].min():.1f} mm "
      "(clearance is a soft penalty in J, so the optimum may sit a little under 15)")

**σ = +1 reproduces SF §4.6** (rms ≈ 2.7 mm, worst ≈ 4.4 mm). σ = −1 is the other assembly of the same links.
It is a different machine whose foot is nowhere near the floor. So for this leg (and for your optimizer) use
`sigma=+1`, and keep it fixed. Note that the optimizer never switches branches by itself. If you want to explore the
other assembly, run a separate set of starts with `sigma=-1`.

## 6. Picture: the leg at mid-stance and its foot path

In [ ]:
def joints(v, dth_one, sigma=+1):
    # O2, A, B, O4, foot in the body frame at one crank travel.
    a, b, c, d, p, delta, th1, th0, hX, hY = v
    H, R = hX + 1j * hY, np.exp(1j * th1)
    th2 = th0 - dth_one
    th3 = theta3(a, b, c, d, th2, sigma)
    A = a * np.exp(1j * th2)
    B = A + b * np.exp(1j * th3)
    P = A + p * np.exp(1j * (th3 + delta))
    return [H + R * z for z in (0, A, B, d, P)]

O2, A, B, O4, P = joints(v_best, 90 * D2R)
C_turn, _, _ = foot(v_best, np.linspace(0, 2 * np.pi, 721))
C_tgt, _, _ = foot(v_best, dth)

fig, ax = plt.subplots(figsize=(7, 6))
ax.axhline(0, color="0.6", lw=1)
ax.plot(C_turn.real, C_turn.imag, color="tab:purple", lw=1.5, label="foot path, one crank turn")
ax.plot(T.real, T.imag, "o", mfc="none", mec="crimson", ms=9, label="targets $T_k$")
ax.plot(C_tgt.real, C_tgt.imag, ".", color="tab:blue", ms=8, label="foot at $\\Delta\\theta_k$")
ax.fill([A.real, B.real, P.real], [A.imag, B.imag, P.imag], color="tab:blue", alpha=0.15)
ax.plot([O2.real, A.real], [O2.imag, A.imag], color="tab:orange", lw=3, label="crank a")
ax.plot([A.real, B.real], [A.imag, B.imag], color="tab:blue", lw=3, label="coupler b")
ax.plot([O4.real, B.real], [O4.imag, B.imag], color="tab:green", lw=3, label="rocker c")
ax.plot([O2.real, O4.real], [O2.imag, O4.imag], "k--", lw=1, label="ground d")
ax.plot([O2.real, O4.real], [O2.imag, O4.imag], "k^", ms=9)
ax.set_aspect("equal"); ax.set_xlabel("X [mm]"); ax.set_ylabel("Y [mm]")
ax.set_title("SF §4.6 best leg at mid-stance (σ = +1)")
ax.legend(loc="lower right", fontsize=8)
plt.show()

## 7. How this plugs into the residuals (SF §4.3)

With `foot` defined, the stance part of the residual vector is just:

```python
C, _, _ = foot(v, dth)                 # 7 complex foot positions
miss = (C - T) / np.sqrt(7)            # 7 complex misses
r_stance = np.concatenate([miss.real, miss.imag])   # 14 real residuals
```

Two practical notes for the optimizer:

1. **NaN means "cannot assemble".** If any θ₃ is NaN, the candidate is not a valid linkage at that crank angle.
   Reject it the way SF does, with a constant large residual (or a sloped penalty, SF §4.6), before the NaNs reach
   `least_squares`.
2. **Angles in radians.** `np.exp(1j*θ)` expects radians. Keep δ, θ₁, θ₀ in radians inside v, or convert at the top
   of `foot` (SF §2.6, "Where students slip").